# Async Python for AI Agents — Minimal Demo

**Scene 7** of *I Made My AI Agent 3× Faster With One Python Keyword (2026)*.

No LLM, no agent framework — just three fake API calls (2 seconds each).

| Mode | Tool phase | Why |
|------|------------|-----|
| Sync | ~6 s | One after another |
| Async | ~2 s | All three in parallel |

Read the markdown cells **before** you run the code. They define `async def`, `await`, and show ASCII flow diagrams you can sketch on camera.

## 1. Three keywords (memorize this table)

| Keyword | What it means | Plain English |
|---------|---------------|---------------|
| `def` | Normal function | "Do this now, block until done." |
| `async def` | Defines a **coroutine** | "This function *can* pause and let others run." |
| `await` | Pause **this** coroutine only | "I'm waiting on I/O — run something else meanwhile." |

**Rule:** You can only use `await` **inside** a function defined with `async def`.

**Rule:** Calling `async def foo()` does **not** run `foo` yet — it returns a coroutine object. Something must **drive** it (e.g. `await foo()` or `asyncio.run(foo())`).

## 2. Sync flow — ASCII timeline (draw this on screen)

One thread. One task at a time. While `time.sleep(2)` runs, **nothing else** in your program moves.

```
TIME ──────────────────────────────────────────────────────────────►

main()
  │
  ├─► fetch("weather")     [██████ sleep 2s ██████]  return
  │
  ├─► fetch("attractions")              [██████ sleep 2s ██████]  return
  │
  └─► fetch("news")                                  [██████ sleep 2s ██████]

TOTAL ≈ 6 seconds   (2 + 2 + 2, stacked)
```

**Call stack mental model:**

```
  main
   └── fetch(weather)  ──blocks──► sleep ──► done
   └── fetch(attractions) ──blocks──► sleep ──► done
   └── fetch(news) ──blocks──► sleep ──► done
```

In [2]:
import asyncio
import time

## 3. Sync code — regular `def` + blocking `time.sleep`

In [6]:
def fetch_weather(city):
    print(f"Fetching weather for {city}")
    time.sleep(2)
    return f"Weather data for {city}"

cities = ["Paris", "London", "Berlin"]

start = time.time()

for city in cities:
    result = fetch_weather(city)
    print(result)

print("Total time:", round(time.time() - start, 2))

Fetching weather for Paris
Weather data for Paris
Fetching weather for London
Weather data for London
Fetching weather for Berlin
Weather data for Berlin
Total time: 6.01


## 4. What `async def` and `await` actually do

### `async def` — you are writing a coroutine

```python
async def fetch_async(name: str) -> str:
    ...
```

- Looks like `def`, but the function body can **yield control** at `await` points.
- `fetch_async("weather")` returns a **coroutine object** — not the string yet.

### `await` — pause here, don't block the whole program

```python
await asyncio.sleep(2)
```

- Means: "This task is waiting on I/O (or a timer). **Event loop**, please run other ready tasks."
- Replaces `time.sleep(2)` in async code — same 2 seconds on the clock, but other coroutines can run during those 2 seconds.

### Who runs the coroutines? The **event loop**

In a notebook you can top-level `await main()`. In a `.py` script you often use `asyncio.run(main())` — that creates a loop, runs `main`, then shuts down.

## 5. Async flow — ASCII timeline (draw this next to sync)

Same three fetches. While one coroutine is at `await sleep`, the others **start**.

```
TIME ──────────────────────────────────────────────────────────────►

event loop (asyncio)
  │
  ├─► fetch_async("weather")     [██████ await sleep ██████]
  ├─► fetch_async("attractions") [██████ await sleep ██████]   ← same 2s window
  └─► fetch_async("news")        [██████ await sleep ██████]   ← overlapped

TOTAL ≈ 2 seconds   (longest single wait, not sum)
```

**Juggling mental model** (the "chef with 3 burners"):

```
       event loop (chef)
            │
    ┌───────┼───────┐
    ▼       ▼       ▼
  task A  task B  task C     all waiting on sleep at the same time
    │       │       │
    └───────┴───────┘
            ▼
      all finish ~2s
```

**Call stack at one instant** (simplified):

```
  main (coroutine)
   └── asyncio.gather
         ├── fetch_async(weather)  ──await sleep──► (paused, not blocking B/C)
         ├── fetch_async(attractions) ──await sleep──►
         └── fetch_async(news) ──await sleep──►
```

## 6. Micro-example — `async def` without `await` doesn't run the body

Run the cell below. Notice: calling `hello()` only **creates** a coroutine; `await hello()` actually runs it.

In [9]:
async def hello():
    print("inside hello()")
    return "done"

coro = hello()  # NOT run yet
print(type(coro), "← coroutine object, body not executed yet")

result = await hello()  # NOW it runs
print("returned:", result)

<class 'coroutine'> ← coroutine object, body not executed yet
inside hello()
returned: done


/var/folders/jw/7pnq01ns5fdbp2gbkfz_n6sm0000gn/T/ipykernel_83872/1838072553.py:5: RuntimeWarning: coroutine 'hello' was never awaited
  coro = hello()  # NOT run yet


## 7. `asyncio.gather` — fan-out / fan-in (ASCII)

`gather` schedules multiple coroutines and waits until **all** finish.

```
                    await asyncio.gather(A, B, C)
                              │
              ┌───────────────┼───────────────┐
              ▼               ▼               ▼
           task A           task B           task C
         await sleep       await sleep       await sleep
              │               │               │
              └───────────────┴───────────────┘
                              ▼
                    results = [rA, rB, rC]
```

This is the **one keyword** from the video title — it is how the agent fires every tool at once.

## 8. Async code — `async def` + `await` + `asyncio.gather`

In [5]:
async def fetch_async(name: str) -> str:
    """Coroutine: pauses at await; other coroutines can run meanwhile."""
    await asyncio.sleep(2)  # non-blocking wait
    return f"got {name}"


async def main():
    t0 = time.perf_counter()
    results = await asyncio.gather(
        fetch_async("weather"),
        fetch_async("attractions"),
        fetch_async("news"),
    )
    elapsed = time.perf_counter() - t0
    print(results)
    print(f"\n⏱  async total: {elapsed:.2f}s  (expect ~2s)")


await main()  # in .py scripts: asyncio.run(main())

['got weather', 'got attractions', 'got news']

⏱  async total: 2.00s  (expect ~2s)


## 9. Side-by-side diff (one frame for the video)

**Sync — blocking**
```python
def fetch(name):
    time.sleep(2)            # freezes the thread
    return f"got {name}"

results = [fetch(n) for n in (...)]
```

**Async — cooperative multitasking**
```python
async def fetch_async(name):
    await asyncio.sleep(2)   # yields to the event loop
    return f"got {name}"

results = await asyncio.gather(
    fetch_async("a"), fetch_async("b"), fetch_async("c"),
)
```

| # | Change | Why |
|---|--------|-----|
| 1 | `def` → `async def` | Function can pause at `await` |
| 2 | `time.sleep` → `await asyncio.sleep` | Don't block other tasks while waiting |
| 3 | loop → `await asyncio.gather(...)` | Start all waits together |

Same outputs. Different **scheduling**. That is the whole lesson before we open `agent_core.py`.

## 10. Bridge to the agent (what changes in production)

In `agent_core.py`, the agent loop does not change the LLM calls — only **tool execution**:

```
# SYNC — tools one by one
for tc in tool_calls:
    result = _execute_tool_sync(...)

# ASYNC — same tools, one gather
results = await asyncio.gather(*[
    _execute_tool_async(...) for tc in tool_calls
])
```

```
  LLM plan tools          tool phase                    LLM answer
  ──────────────          ──────────                    ──────────
  sync:   [LLM ~5s]  →  [t1][t2][t3]...[t10]  →  [LLM ~5s]   (~30s total)
  async:  [LLM ~5s]  →  [all tools in parallel ~2s]  →  [LLM ~5s]   (~12s total)
```

Next: run `sync_agent.py` and `async_agent.py` side by side with the **5-city demo prompt** from the sidebar.